# Category-Driven Model Notebook

카테고리별로 선택한 핵심 피처(2~4개)만 사용하여
결측/이상치 처리를 수행하고 GroupKFold + LightGBM으로 예측합니다.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

BASE_DIR = Path('..')
DATA_DIR = BASE_DIR / 'data'
CODE_DIR = BASE_DIR / 'code'

TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
LAYOUT_PATH = DATA_DIR / 'layout_info.csv'
TARGET = 'avg_delay_minutes_next_30m'


In [5]:
# 카테고리별 추천 피처(각 2~4개)
feature_groups = {
    '배터리_충전': ['low_battery_ratio', 'charge_queue_length', 'avg_charge_wait'],
    '혼잡_충돌': ['congestion_score', 'max_zone_density', 'near_collision_15m'],
    '주문_수요': ['order_inflow_15m', 'unique_sku_15m', 'urgent_order_ratio', 'sku_concentration'],
    '로봇_AGV': ['robot_utilization', 'robot_charging', 'task_reassign_15m', 'agv_task_success_rate'],
    'KPI_성과': ['kpi_otd_pct', 'sort_accuracy_pct', 'manual_override_ratio'],
    '인력_안전': ['staff_on_floor', 'forklift_active_count', 'safety_score_monthly'],
    '물류_재고': ['loading_dock_util', 'staging_area_util', 'inventory_turnover_rate', 'backorder_ratio'],
    '환경_시설': ['warehouse_temp_avg', 'humidity_pct', 'layout_compactness', 'maintenance_schedule_score'],
    '외부_날씨': ['external_temp_c', 'precipitation_mm'],
    '인프라': ['wms_response_time_ms', 'network_latency_ms', 'scanner_error_rate'],
}

selected_features = []
for _, cols in feature_groups.items():
    selected_features.extend(cols)
selected_features = list(dict.fromkeys(selected_features))

feature_map_rows = []
for g, cols in feature_groups.items():
    for c in cols:
        feature_map_rows.append({'group': g, 'column': c})

feature_map_df = pd.DataFrame(feature_map_rows)
feature_map_df.to_csv(CODE_DIR / 'category_feature_map.csv', index=False)
print('selected feature count:', len(selected_features))
feature_map_df.head()


selected feature count: 33


,group,column
0,배터리_충전,low_battery_ratio
1,배터리_충전,charge_queue_length
2,배터리_충전,avg_charge_wait
3,혼잡_충돌,congestion_score
4,혼잡_충돌,max_zone_density


In [6]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
layout = pd.read_csv(LAYOUT_PATH)

train = train.merge(layout, on='layout_id', how='left')
test = test.merge(layout, on='layout_id', how='left')

available_features = [c for c in selected_features if c in train.columns]
missing_features = [c for c in selected_features if c not in train.columns]
print('available:', len(available_features), '| missing:', len(missing_features))
if missing_features:
    print('missing features:', missing_features)

X_train = train[available_features].copy()
X_test = test[available_features].copy()
y = train[TARGET].copy()
groups = train['scenario_id'].copy()
test_id = test['ID'].copy()

print('X_train:', X_train.shape, 'X_test:', X_test.shape)


available: 33 | missing: 0
X_train: (250000, 33) X_test: (50000, 33)


In [7]:
# 결측치 처리: 중앙값 + 결측 indicator
missing_ratio = X_train.isna().mean()
indicator_cols = missing_ratio[missing_ratio >= 0.08].index.tolist()

for c in indicator_cols:
    X_train[f'is_null_{c}'] = X_train[c].isna().astype(np.int8)
    X_test[f'is_null_{c}'] = X_test[c].isna().astype(np.int8)

med = X_train.median(numeric_only=True)
X_train = X_train.fillna(med)
X_test = X_test.fillna(med)

# 이상치 처리: 0.5% ~ 99.5% 클리핑
lower = X_train.quantile(0.005)
upper = X_train.quantile(0.995)
X_train = X_train.clip(lower=lower, upper=upper, axis=1)
X_test = X_test.clip(lower=lower, upper=upper, axis=1)

# 긴 꼬리 일부 변수 log1p
log_cols = [c for c in ['order_inflow_15m', 'charge_queue_length', 'avg_charge_wait'] if c in X_train.columns]
for c in log_cols:
    X_train[f'log_{c}'] = np.log1p(np.clip(X_train[c], a_min=0, a_max=None))
    X_test[f'log_{c}'] = np.log1p(np.clip(X_test[c], a_min=0, a_max=None))

print('indicator cols:', len(indicator_cols))
print('final feature count:', X_train.shape[1])


indicator cols: 30
final feature count: 66


In [ ]:
# GroupKFold + LightGBM
N_SPLITS = 5
SEEDS = [42, 2026]

gkf = GroupKFold(n_splits=N_SPLITS)
oofs = []
preds = []
scores = []

base_params = dict(
    objective='mae',
    metric='l1',
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=63,
    min_child_samples=60,
    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.85,
    reg_alpha=0.05,
    reg_lambda=1.0,
    n_jobs=-1,
)

for seed in SEEDS:
    print(f'===== seed {seed} =====')
    oof = np.zeros(len(X_train), dtype=np.float64)
    pred_test = np.zeros(len(X_test), dtype=np.float64)

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train, y, groups), 1):
        X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        model = LGBMRegressor(**base_params, random_state=seed)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric='l1',
            callbacks=[early_stopping(150), log_evaluation(200)]
        )

        oof[va_idx] = model.predict(X_va, num_iteration=model.best_iteration_)
        pred_test += model.predict(X_test, num_iteration=model.best_iteration_) / N_SPLITS

    score = mean_absolute_error(y, oof)
    print(f'seed {seed} OOF MAE: {score:.5f}')

    oofs.append(oof)
    preds.append(pred_test)
    scores.append(score)

weights = 1 / np.array(scores)
weights = weights / weights.sum()

blend_oof = np.zeros(len(X_train), dtype=np.float64)
blend_test = np.zeros(len(X_test), dtype=np.float64)
for w, oof, pred in zip(weights, oofs, preds):
    blend_oof += w * oof
    blend_test += w * pred

blend_score = mean_absolute_error(y, blend_oof)
print('seed scores:', [round(s, 5) for s in scores])
print('blend OOF MAE:', round(blend_score, 5))


In [ ]:
sub = pd.DataFrame({
    'ID': test_id,
    TARGET: blend_test
}).sort_values('ID').reset_index(drop=True)

out_path = CODE_DIR / 'submission_category.csv'
sub.to_csv(out_path, index=False)

print('saved:', out_path)
sub.head()
